In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ========================================
# CONFIGURATION
# ========================================

BASE_TOPIC_URL = "https://www.rappler.com/topic/agriculture/page/{}/"
START_PAGE = 1
END_PAGE = 5  # adjust after confirming how many pages exist
OUTPUT_FILE = "rappler_agriculture.csv"

# Initialize DataFrame
corpus = pd.DataFrame(columns=['title', 'link', 'date_published', 'author', 'text'])


# ========================================
# SCRAPER FUNCTIONS
# ========================================

def get_article_details(article_url):
    """Extract title, date, author, and text from an article."""
    try:
        res = requests.get(article_url, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")

        # Title
        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else None

        # Date
        date_tag = soup.find("time")
        date_published = date_tag.get("datetime") if date_tag else None

        # Author
        author_tag = soup.find("a", rel="author") or soup.find("span", class_="author__name")
        author = author_tag.get_text(strip=True) if author_tag else None

        # Article text
        paragraphs = soup.select("article p")
        text = " ".join(p.get_text(strip=True) for p in paragraphs) if paragraphs else None

        return [title, article_url, date_published, author, text]

    except Exception as e:
        print(f"Error scraping article {article_url}: {e}")
        return [None, article_url, None, None, None]


def scrape_topic_page(page_number):
    """Scrapes one page of the Agriculture topic."""
    url = BASE_TOPIC_URL.format(page_number)
    print(f"Scraping topic page {page_number}: {url}")

    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
    except Exception as e:
        print(f"Failed to load topic page {page_number}: {e}")
        return []

    articles_data = []

    # On archive pages, each article link is inside <h2><a href=...>
    article_links = [a['href'] for a in soup.select("h2 > a[href]")]
    if not article_links:
        print(f"No articles found on page {page_number}.")
        return []

    for link in article_links:
        if not link.startswith("http"):
            link = "https://www.rappler.com" + link

        article_details = get_article_details(link)
        articles_data.append(article_details)
        print(f"Scraped: {article_details[0]}")
        time.sleep(1)  # polite delay

    return articles_data


# ========================================
# MAIN LOOP
# ========================================

all_data = []

for page in range(START_PAGE, END_PAGE + 1):
    page_data = scrape_topic_page(page)
    if not page_data:
        print(f"Stopping at page {page} (no more articles).")
        break
    all_data.extend(page_data)

# Combine into DataFrame
corpus = pd.DataFrame(all_data, columns=['title', 'link', 'date_published', 'author', 'text'])

# Save to CSV
corpus.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Scraping complete. {len(corpus)} articles saved to {OUTPUT_FILE}.")


Scraping topic page 1: https://www.rappler.com/topic/agriculture/page/1/
Scraped: Arms to farms: Maguindanao del Sur’s agriculture roadmap for former rebels
Scraped: Chinese smugglers ‘operate freely’ with help of PH gov’t officials — Pangilinan
Scraped: Bounty Fresh aims to strengthen operations amid growing appetite for poultry
Scraped: Century Pacific buys another coconut facility. What’s next for its food business?
Scraped: [Ask the Tax Whiz] How do Filipinos benefit from a higher excise tax on sugary drinks?
Scraped: DA to take over farm-to-market road projects from DPWH
Scraped: Zaldy Co-linked firm ranks No. 3 in 2024 farm-to-market road projects
Scraped: Trump’s tariff war is ASEAN ministers’ lunchtime talk: ‘Have you closed?’
Scraped: Cheng family, owner of Bounty Fresh, now has 200 restaurants via The Bistro Group
Scraping topic page 2: https://www.rappler.com/topic/agriculture/page/2/
Scraped: NFA needs P3 billion for emergency palay procurement
Scraped: DAR eyes P17.4-billi